# Deep Agents — Multi-Subagent Orchestration

This notebook demonstrates how to design, instantiate, and orchestrate **Subagents** within the `deepagents` and LangGraph framework.

### Architecture Overview
In `deepagents`, the supervisor agent acts as the main orchestrator equipped with a built-in `task` delegation tool. When a query is submitted:
1. The **Supervisor Agent** reviews the registered subagents in `subagents=[...]`.
2. If a request matches a subagent's `description`, the supervisor delegates the task using `task(subagent_name, prompt)`.
3. The **Subagent** executes the assigned task using its own specialized tools, instructions, and optional model override.
4. The **Subagent** returns its results back to the supervisor.
5. The **Supervisor** synthesizes the findings and delivers the complete, coherent answer to the user.

In [ ]:
import os
from typing import Literal
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import MemorySaver
from deepagents import create_deep_agent, SubAgent, CompiledSubAgent
from tavily import TavilyClient

# Load environment variables
load_dotenv()

# Tracing and LangSmith Configuration
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGSMITH_API_KEY", "")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGSMITH_PROJECT", "CLANICAL AGENT")

# API Keys
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY", "")
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY", "")

# Initialize primary LLM (Groq models support fast and reliable function/tool calling)
llm = ChatGroq(model="openai/gpt-oss-120b")
# Alternative fast model:
# llm = ChatGroq(model="llama-3.3-70b-versatile")

print("Environment loaded and LLM initialized successfully.")

## Step 1: Define Specialized Tools
Subagents can each have their own dedicated tools. Here we configure:
- `internet_search`: Powered by Tavily for real-time web research.
- `financial_metric_calculator`: Computes profitability, gross margins, and financial metrics.

In [ ]:
# Initialize Tavily search client
tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

def internet_search(
    query: str,
    max_results: int = 5,
    topic: Literal["general", "news", "finance"] = "general",
    include_raw_content: bool = False,
):
    """Run an internet web search to retrieve up-to-date facts, current news, and real-time data."""
    return tavily_client.search(
        query,
        max_results=max_results,
        include_raw_content=include_raw_content,
        topic=topic,
    )

def financial_metric_calculator(revenue: float, cost: float) -> dict:
    """Calculate gross profit, gross margin percentage, and net return."""
    profit = revenue - cost
    margin = (profit / revenue * 100) if revenue > 0 else 0.0
    return {
        "revenue": revenue,
        "cost": cost,
        "gross_profit": round(profit, 2),
        "gross_margin_pct": f"{round(margin, 2)}%"
    }

print("Tools registered: internet_search, financial_metric_calculator")

## Pattern 1: Declarative Subagents (`SubAgent` TypedDict)
The standard and recommended way in `deepagents` is defining subagents as declarative dictionaries conforming to the `SubAgent` specification.

Each specification requires:
- `name`: Unique identifier used by the supervisor to delegate work.
- `description`: Explains what the subagent specializes in. The supervisor uses this to decide when to delegate.
- `system_prompt`: Specific instructions and persona for the subagent.
- `tools`: Specialized tools available to this subagent.
- `model` *(optional)*: Dedicated LLM instance if overriding the main agent model.

In [ ]:
# 1. Research Subagent: Focused on web search and real-time facts
research_subagent: SubAgent = {
    "name": "research_agent",
    "description": (
        "Specialist in searching the web for current events, live market news, and verified facts. "
        "Delegate tasks here whenever up-to-date internet research is needed."
    ),
    "system_prompt": (
        "You are an expert research analyst. Use the internet_search tool to gather verified, "
        "fact-checked information. Summarize findings clearly with key data points."
    ),
    "tools": [internet_search],
    "model": llm,
}

# 2. Financial Analyst Subagent: Focused on financial strategy and business models
financial_analyst_subagent: SubAgent = {
    "name": "financial_analyst",
    "description": (
        "Specialist in business strategies, commercialization, revenue models, and financial analysis. "
        "Delegate tasks here when strategic business breakdown or financial metric calculation is required."
    ),
    "system_prompt": (
        "You are an experienced Chief Financial Officer (CFO) and financial strategist. "
        "Analyze revenue streams, cost structures, and strategic levers to provide actionable business recommendations."
    ),
    "tools": [financial_metric_calculator],
    "model": llm,
}

subagents_list = [research_subagent, financial_analyst_subagent]
print(f"Configured {len(subagents_list)} subagents: {[s['name'] for s in subagents_list]}")

## Step 2: Instantiate the Supervisor Agent
The supervisor orchestrates the subagents using its built-in `task` delegation tool. We attach a `MemorySaver` checkpointer for session state persistence.

In [ ]:
# Instantiate the main supervisor agent
supervisor_agent = create_deep_agent(
    model=llm,
    name="Supervisor",
    system_prompt=(
        "You are the Lead Strategic Supervisor. Your role is to coordinate and solve user requests by orchestrating your specialized subagents:\n"
        "- Delegate live web queries, market news, and internet research to 'research_agent'.\n"
        "- Delegate business strategy, revenue models, and financial planning to 'financial_analyst'.\n"
        "After receiving responses from the subagents, synthesize them into a clear, professional, and well-structured answer."
    ),
    subagents=subagents_list,
    checkpointer=MemorySaver(),
)

print("Supervisor agent ready with subagents connected!")

## Step 3: Invoke the Supervisor Agent

> **Note on Input Format:** In `deepagents`, user input must be passed via the `messages` key:
> `{"messages": [{"role": "user", "content": "..."}]}` or `{"messages": [("user", "...")]}`.
> Provide `config={"configurable": {"thread_id": "..."}}` to maintain conversation state.

In [ ]:
# Configure thread ID for conversation persistence
thread_config = {"configurable": {"thread_id": "openai_finance_research_01"}}

user_query = "What is the financial strategy of OpenAI nowadays? Search the internet for recent news and analyze their monetization model."

response = supervisor_agent.invoke(
    {"messages": [{"role": "user", "content": user_query}]},
    config=thread_config
)

print("=== SUPERVISOR FINAL RESPONSE ===\n")
print(response["messages"][-1].content)

## Step 4: Multi-Turn Conversation (Persistent Memory)
Using the same `thread_id`, the supervisor remembers all previous context, tool calls, and subagent answers.

In [ ]:
followup_query = "Based on that strategy, what are their main revenue drivers and financial risks?"

response_followup = supervisor_agent.invoke(
    {"messages": [{"role": "user", "content": followup_query}]},
    config=thread_config
)

print("=== FOLLOW-UP RESPONSE ===\n")
print(response_followup["messages"][-1].content)

## Pattern 2: Pre-compiled Subagents (`CompiledSubAgent`)
If you have an existing pre-compiled agent or LangGraph graph (from `create_deep_agent(...)` or custom `StateGraph`), you can integrate it using `CompiledSubAgent`.

```python
compiled_subagent: CompiledSubAgent = {
    "name": "market_researcher",
    "description": "Researches market trends and competitors",
    "runnable": compiled_agent_graph  # Compiled StateGraph
}
```

In [ ]:
# 1. Compile a standalone subagent graph
compiled_analyst_graph = create_deep_agent(
    model=llm,
    system_prompt="You are a SaaS Growth Strategist. Analyze customer acquisition costs, churn, and sales strategies.",
    tools=[financial_metric_calculator],
)

# 2. Package into a CompiledSubAgent dictionary
saas_analyst_subagent: CompiledSubAgent = {
    "name": "saas_growth_analyst",
    "description": "Specialist in analyzing SaaS business growth, sales funnels, and revenue metrics.",
    "runnable": compiled_analyst_graph,
}

# 3. Create a supervisor with the compiled subagent
compiled_supervisor = create_deep_agent(
    model=llm,
    system_prompt="You are the Executive Director. Delegate SaaS growth questions to 'saas_growth_analyst'.",
    subagents=[saas_analyst_subagent],
    checkpointer=MemorySaver(),
)

print("CompiledSubAgent supervisor initialized successfully!")

## Step 5: Streaming Execution to Inspect Subagent Delegation
You can inspect subagent tool calls and intermediate delegation steps in real time using `.stream()`.

In [ ]:
streaming_config = {"configurable": {"thread_id": "streaming_thread_demo"}}
query = "Search the best finance strategy to achieve the highest sales on the internet."

print("Streaming execution trace:\n")
for chunk in supervisor_agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    config=streaming_config,
    stream_mode="values"
):
    msg = chunk["messages"][-1]
    msg_class = type(msg).__name__
    
    if msg_class == "AIMessage" and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"[Supervisor Action] Delegating to tool/subagent -> {tc['name']} with args: {tc['args']}")
    elif msg_class == "ToolMessage":
        preview = str(msg.content)[:120].replace("\n", " ")
        print(f"[Tool/Subagent Output] ({msg.name}) -> {preview}...")
    elif msg_class == "AIMessage" and msg.content and not msg.tool_calls:
        print(f"\n[Final Response Preview] -> {msg.content[:200]}...")

## Key Takeaways
- **Declarative SubAgent**: Clean, dictionary-based definition (`name`, `description`, `system_prompt`, `tools`, `model`).
- **CompiledSubAgent**: Allows pre-compiled `create_deep_agent` graphs or custom LangGraph runnables to be used as subagents.
- **Built-in `task` Tool**: The supervisor delegates automatically based on the subagent `description`.
- **Message Schema**: Input must always be formatted as `{"messages": [{"role": "user", "content": "..."}]}`.
- **Thread Isolation**: Use `{"configurable": {"thread_id": "..."}}` with `MemorySaver()` for persistent multi-turn conversations.